# Choice of scaling

In [ ]:
from functools import partial
from lucifex.sim import parallel_run, as_grid_simulation
from lucifex.plt import plot_colormap, plot_line, set_ipynb_variable
from lucifex.utils.array_utils import as_index
from crocodil.theory.system_a import threshold_rayleigh
from crocodil.dns.system_a import dns_system_a, SYSTEM_A_REFERENCE

scaling_opts = ('advective', 'diffusive')
NX = 80
NY= 80
COURANT_ADV = 0.75
COURANT_DIFF = 0.75
COURANT_REAC = 0.1

n_proc = set_ipynb_variable('N_PROC', 3)
n_stop = set_ipynb_variable('N_STOP', 200)
dt_init = 1e-6
n_init = 10

Ra_thresh = threshold_rayleigh(SYSTEM_A_REFERENCE['aspect'], 1.0, NY, 2)
print(f"Ra = {SYSTEM_A_REFERENCE['Ra']} , Ra_thresh = {Ra_thresh}")

STORE = 1
create_sim = dns_system_a(store_delta=STORE)

simulations = parallel_run(
    create_sim, n_proc, n_stop, 
    dt_init=dt_init, n_init=n_init,
    serialize=partial(as_grid_simulation, slc_func='::5', include=('c', 'cMinMax', 'uRMS', 'uMinMax')),
    link=True,
    max_nbytes=None,
    mmap_mode=None,
)(
    Nx=NX,
    Ny=NY,
    **SYSTEM_A_REFERENCE,
    courant_adv=COURANT_ADV,
    courant_diff=COURANT_DIFF,
    courant_reac=COURANT_REAC,
    c_stabilization=None,
    c_limits=True,
    diagnostic=True,
)(
    scaling=scaling_opts,
)